# R-tag pipeline — single run walkthrough

End-to-end **KB331** sample: filter GDS → per-resonator RTEG layouts with MBE/MTE routing.

**Docs:** [README](../README.md) · [CLAUDE](../CLAUDE.md) · run top-to-bottom from `python_code/`.

| Step | What it does | Main call |
|------|----------------|-----------|
| 1 | Validate inputs | layermap + file check |
| 2 | Find resonators / vias | `identify` |
| 2.4 | Original-die collar intercepts | `collect_die_collar_intercepts` |
| 3 | Center resonator in GSG PPD | `prep_resonator_ppd` |
| 4 | Place in die frame + export framed GDS | `prep_rteg_in_frame`, `export_gds` |
| 5.1–5.4 | MTE signal: collect → classify → extend → pad stretch | see §5 cells |
| 6.1–6.3 | MBE signal + ground filler | see §6 cells |

**KB331 split (8 resonators):** `center_pad` MTE route → indices **1, 3, 4, 6** · `collar_extend` MBE signal → **0, 2, 5, 7**.


take intercept point 
take angles
from die filter input  

reduce curves to 45 or 22.5 degree

shift resonator in runtime

extend MBE rectangle width box to be 60 microns away from GSG

external ask:
ppd1 port from left or top

In [1]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ORIGNAL_RTEG = DRAFT / "original_rteg"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

Working directory: C:\Users\santosr4\Documents\rTEG Automation\python_code
Input files:       C:\Users\santosr4\Documents\rTEG Automation\python_code\input_files
Draft output:      C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Source code:       C:\Users\santosr4\Documents\rTEG Automation\python_code\src


## Define inputs

Set paths to the filter GDS, die frame, GSG probe template, and Skyworks layermap. All paths are under `input_files/`. The next code cell assigns `FILTER`, `FRAME`, `PPD`, `LAYERMAP`.


In [2]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process inputs

**~30 s read**

Confirms all four input files exist and prints a short inventory (size + role). Also reads the frame template bbox so you know the probe window size before placement.

**Output:** sanity-check table — aborts if anything is missing.


In [3]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}×{fy1 - fy0:.1f} µm"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


,file,path,exists,size,role
0,Filter layout,KB331_N_01.gds,True,"655,360 B",Clean hierarchical filter GDS — resonators + c...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460.0×580.0 µm GSG probe frame (six BAW_MB2 pads)
2,Probe device,GSG_frame.gds,True,"10,240 B",ppd_1port — GSG pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name → GDS (layer, datatype)"


## 2. Selection

**~30 s read**

Prepare for resonator extraction: load the layermap, inspect GDS hierarchy, then identify which instances in the filter become R-tags.

| Sub-step | Module | Purpose |
|----------|--------|---------|
| 2.1 | `layermap.py` | Name ↔ GDS layer/datatype |
| 2.2 | `inspect_refs.py` | Hierarchy / reference listing |
| 2.3 | `separate.py` | Resonator + via identification |
| 2.4 | `rteg_die_intercepts.py` | Original-die MBE/MTE collar intercepts + angles |

**Output:** `layermap`, `res_df`, `res_list`, `via_df`, `identification`, `die_intercepts`.


### 2.1 — `layermap.py`

**~30 s read**

Loads `input_files/layermap` so later steps refer to layers by Skyworks names (`BAW_MBE`, `BAW_MTE`, …) instead of raw GDS numbers.

**Call:** `load_layermap(LAYERMAP)` → `layermap` object with `.pair(name)` lookups.


In [4]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      -> GDS (2, 0)
  BAW_MTE      -> GDS (5, 0)
  BAW_MB2      -> GDS (12, 0)
  BAW_EDGE     -> GDS (9, 0)


### 2.2 — `inspect_refs.py`

**~30 s read**

Walks the filter GDS hierarchy: cell references, labels, and bounding boxes. Quick sanity check that resonator masters and parent cells look like the expected KB331 structure before automated identification.

**Call:** `inspect_gds(FILTER)` (optional for frame/PPD too).


In [5]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")


=== KB331_N_01.gds ===

shuntq3_CDNS_781040784740: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_781040784741: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)  laye

### 2.3 — `separate.py`

**~30 s read**

Finds placed resonators under the filter parent (`series*`, `shunt*`, `rcap*`, `mimcap*` masters) and `vtb` vias. Returns dataframe rows plus live `Resonator` objects with placement transform preserved.

**Call:** `identify(FILTER)` → `identification` with `.resonator_rows()`, `.resonators`, `.via_rows()`.

**Output:** `res_df`, `res_list` — one row/object per R-tag candidate (KB331: 8).


In [6]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

Parent cell: KB331_N_01
Resonators: 8  |  Vias: 4


,index,inst_name,master_name,type,origin_x,origin_y,rotation_deg,split_base
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,282.6,183.1,0.0,None
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,234.0,98.3,180.0,None
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,95.8,145.1,90.0,None
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,92.0,217.4,270.0,None
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,311.9,458.9,90.0,None
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,157.8,412.7,0.0,None
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,307.6,317.6,270.0,None
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,187.7,294.1,270.0,None


### 2.4 — `rteg_die_intercepts.py`

**~30 s read**

Reads `{parent}_connectMTE` / `{parent}_connectMBE` from the filter GDS and records where filter interconnect meets each resonator collar on **MBE and MTE** — mouth intercept points A/B plus entry and mouth angles in **filter-die coordinates** (same frame as `origin_x` / `origin_y`).

NPI requirement: custom RTEG interconnects must land on the same mouth positions and entry angles as the original die layout (mirrors SKILL `{cell}_connect_backup` + collar angle params).

**Call:** `collect_die_collar_intercepts(identification, layermap)` → `die_intercepts`; merge onto step 2.3 rows with `merge_resonator_intercept_rows`.

**Output:** `die_intercepts`, updated `res_df` (same `index` as step 2.3). Steps 5–6 will consume `die_intercepts` / `transform_die_intercept_to_rteg` so routing targets match the filter layout.

In [7]:
from src.rteg_die_intercepts import (
    collect_die_collar_intercepts,
    merge_resonator_intercept_rows,
)

die_intercepts = collect_die_collar_intercepts(identification, layermap)
res_df = pd.DataFrame(
    merge_resonator_intercept_rows(
        identification, layermap, die_intercepts=die_intercepts
    )
)

intercept_cols = [
    "index",
    "inst_name",
    "type",
    "mte_status",
    "mte_intercept_a",
    "mte_intercept_b",
    "mte_entry_angle_deg",
    "mbe_status",
    "mbe_intercept_a",
    "mbe_intercept_b",
    "mbe_entry_angle_deg",
]
print("Original-die collar intercepts (filter coordinates; index matches step 2.3)")
res_df[intercept_cols]

Original-die collar intercepts (filter coordinates; index matches step 2.3)


,index,inst_name,type,mte_status,mte_intercept_a,mte_intercept_b,mte_entry_angle_deg,mbe_status,mbe_intercept_a,mbe_intercept_b,mbe_entry_angle_deg
0,0,shuntq3_CDNS_781040784740,shunt,ok,"(226.55, 119.06)","(344.82, 60.64)",269.1,ok,"(218.99, 245.99)","(360.14, 239.10)",65.2
1,1,shuntq3_CDNS_781040784740,shunt,ok,"(202.37, 223.34)","(284.48, 223.34)",84.0,ok,"(167.91, 173.08)","(317.84, 230.64)",94.4
2,2,seriesq3_CDNS_781040784741,series,ok,"(90.90, 235.64)","(119.45, 239.03)",87.0,ok,"(140.75, 201.93)","(350.79, 239.10)",24.8
3,3,seriesq3_CDNS_781040784741,series,ok,"(218.07, 332.99)","(173.37, 355.12)",49.9,ok,"(138.15, 257.36)","(254.45, 322.37)",38.1
4,4,shuntq3_CDNS_781040784742,shunt,ok,"(184.88, 348.24)","(162.80, 486.86)",197.4,ok,"(186.84, 472.87)","(227.46, 351.34)",198.4
5,5,shuntq3_CDNS_781040784745,shunt,ok,"(273.92, 315.38)","(331.55, 487.43)",353.7,ok,"(112.50, 533.50)","(200.50, 533.50)",88.9
6,6,seriesq3_CDNS_781040784747,series,ok,"(225.43, 244.08)","(196.66, 329.41)",195.7,ok,"(226.50, 330.31)","(278.43, 239.10)",208.8
7,7,seriesq3_CDNS_781040784748,series,ok,"(196.22, 453.45)","(213.20, 436.46)",86.8,ok,"(288.61, 222.34)","(255.07, 304.11)",346.5


### 2.4b — Die intercept debug table (filter coordinates)

Step 2.4 values are in **filter-die world space** (same frame as `origin_x` / `origin_y`). Each mouth also stores **body-local offsets** (`mte_intercept_a_local`, …) relative to the resonator metal bbox center — these are what get replayed in the RTEG frame in steps 5–6.


In [8]:
# All resonators — original-die intercepts (filter coordinates, µm)
die_debug_cols = [
    "index", "inst_name", "type",
    "mte_intercept_a", "mte_intercept_b", "mte_mouth_span_um", "mte_entry_angle_deg", "mte_collar_area_um2",
    "mbe_intercept_a", "mbe_intercept_b", "mbe_mouth_span_um", "mbe_entry_angle_deg", "mbe_collar_area_um2",
]
display(res_df[die_debug_cols])

,index,inst_name,type,mte_intercept_a,mte_intercept_b,mte_mouth_span_um,mte_entry_angle_deg,mte_collar_area_um2,mbe_intercept_a,mbe_intercept_b,mbe_mouth_span_um,mbe_entry_angle_deg,mbe_collar_area_um2
0,0,shuntq3_CDNS_781040784740,shunt,"(226.55, 119.06)","(344.82, 60.64)",131.907,269.1,14551.1,"(218.99, 245.99)","(360.14, 239.10)",141.315,65.2,9889.1
1,1,shuntq3_CDNS_781040784740,shunt,"(202.37, 223.34)","(284.48, 223.34)",82.112,84.0,14551.1,"(167.91, 173.08)","(317.84, 230.64)",160.592,94.4,3182.3
2,2,seriesq3_CDNS_781040784741,series,"(90.90, 235.64)","(119.45, 239.03)",28.753,87.0,3269.4,"(140.75, 201.93)","(350.79, 239.10)",213.302,24.8,15123.5
3,3,seriesq3_CDNS_781040784741,series,"(218.07, 332.99)","(173.37, 355.12)",49.879,49.9,187.0,"(138.15, 257.36)","(254.45, 322.37)",133.230,38.1,4350.0
4,4,shuntq3_CDNS_781040784742,shunt,"(184.88, 348.24)","(162.80, 486.86)",140.373,197.4,16497.4,"(186.84, 472.87)","(227.46, 351.34)",128.136,198.4,17018.0
5,5,shuntq3_CDNS_781040784745,shunt,"(273.92, 315.38)","(331.55, 487.43)",181.448,353.7,5980.7,"(112.50, 533.50)","(200.50, 533.50)",88.000,88.9,4855.0
6,6,seriesq3_CDNS_781040784747,series,"(225.43, 244.08)","(196.66, 329.41)",90.051,195.7,9497.3,"(226.50, 330.31)","(278.43, 239.10)",104.958,208.8,9889.1
7,7,seriesq3_CDNS_781040784748,series,"(196.22, 453.45)","(213.20, 436.46)",24.016,86.8,16497.4,"(288.61, 222.34)","(255.07, 304.11)",88.383,346.5,4350.0


In [9]:
via_df

,index,master_name,origin_x,origin_y,rotation_deg
0,0,vtb3_CDNS_781040784743,208.6,127.4,270.0
1,1,vtb3_CDNS_781040784746,335.0,158.9,270.0
2,2,vtb3_CDNS_781040784749,122.6,176.9,270.0
3,3,vtb3_CDNS_781040784749,65.2,185.8,90.0


## 3. Separation

**~30 s read**

For each identified resonator, build an in-memory **PPD + resonator** assembly: GSG probe frame from `GSG_frame.gds` with the resonator centered and clearance-adjusted inside it. No die frame yet — that is step 4.

**Output:** `ppd_assemblies` — one per resonator, ready for frame placement.


### 3 — PPD + orientation placement

**~30 s read**

**Calls:** `prep_resonator_ppd` (with `identification` + `layermap`) → optional orientation analysis.

1. **Center** resonator bbox on the PPD template.
2. **Clearance nudge** — ≥10 µm to GSG pad metal, ≥6 µm to release holes.
3. **Orientation shift** — small search to maximize pad clearance while keeping DRC-friendly placement.

**Output:** `ppd_assemblies[index].top_cell` — PPD ref + resonator ref in a scratch cell.


In [10]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


,index,inst_name,master_name,type,ppd_origin_x,ppd_origin_y,resonator_origin_x,resonator_origin_y,centering_shift_x,centering_shift_y,clearance_shift_x,clearance_shift_y,orientation_shift_x,orientation_shift_y,min_release_clearance_um,shift_x,shift_y,assembly_center_x,assembly_center_y
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,498.6,222.8,105.2,59.7,90.8,0.0,20.0,-20.0,37.2,216.0,39.7,388.2,245.3
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,500.3,222.8,153.8,144.5,92.5,0.0,20.0,-20.0,34.1,266.3,124.5,388.2,245.3
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,414.4,245.3,289.5,100.2,19.0,0.0,10.0,0.0,32.8,318.6,100.2,388.2,245.3
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,412.4,245.3,293.3,27.9,17.0,0.0,10.0,0.0,31.5,320.4,27.9,388.2,245.3
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,0.0,0.0,468.7,245.8,89.0,-213.1,47.8,0.0,20.0,0.0,32.9,156.8,-213.1,388.2,245.3
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,0.0,0.0,475.8,221.8,230.1,-180.9,77.9,0.0,10.0,-10.0,30.8,318.0,-190.9,388.2,245.3
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,0.0,0.0,423.7,225.3,78.8,-72.3,17.3,0.0,20.0,-20.0,37.8,116.1,-92.3,388.2,245.3
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,0.0,0.0,448.0,235.6,193.4,-48.5,56.9,0.0,10.0,-10.0,31.8,260.3,-58.5,388.2,245.3


### 3 — Preview (optional)

**~30 s read**

Optional SVG grid of each PPD+resonator assembly. Use to spot bad centering or pad collisions before die-frame placement. Code is commented out by default.


In [11]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting up

**~30 s read**

Place each PPD assembly into the **die frame template** (`KB331_N_Frame.gds`) and add the right-side MBE width filler. Margins are measured from the inner MBE ring cavity (not the outer 460×580 µm bbox).

- **X:** PPD/GSG frame left-aligned at 4% inner margin; wide resonators may get a **resonator-only** left shift (5 µm clearance to filler right edge).
- **Y:** assembly centered in 7% vertical margin band.
- **Filler:** MBE rectangle from inner-frame center line → right margin, full assembly height.

**Output:** `rteg_assemblies` — frame + placed PPD/resonator + filler per index.


### 4 — Die frame placement

**~30 s read**

**Call:** `prep_rteg_in_frame(ppd_assemblies, FRAME)` → `rteg_assemblies`

PPD and resonator are placed as **separate references** so only the resonator moves when enforcing filler clearance (`resonator_frame_shift` on the assembly object).

Also exports **framed-only** GDS to `draft_output/original_rteg/` via `export_gds` (geometry through step 4, no routing metal yet).


In [12]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

Built 8 RTEG frame assemblies


C:\Users\santosr4\AppData\Local\Temp\ipykernel_12528\573027339.py:7: UserWarning: Assembly [0] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
C:\Users\santosr4\AppData\Local\Temp\ipykernel_12528\573027339.py:7: UserWarning: Assembly [1] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)


,index,inst_name,type,frame_origin_x,frame_origin_y,assembly_origin_x,assembly_origin_y,resonator_frame_shift_x,resonator_frame_shift_y,content_center_x,...,inner_frame_x1,inner_frame_y1,mbe_filler_x0,mbe_filler_y0,mbe_filler_x1,mbe_filler_y1,inner_content_width,mbe_filler_width,x_margin_pct,y_margin_pct
0,0,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,-7.8,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
1,1,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,-11.5,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
2,2,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
3,3,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
4,4,shuntq3_CDNS_781040784742,shunt,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
5,5,shuntq3_CDNS_781040784745,shunt,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
6,6,seriesq3_CDNS_781040784747,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
7,7,seriesq3_CDNS_781040784748,series,0.0,0.0,-216.2,44.7,0.0,0.0,230.0,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07


In [13]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [14]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        ORIGNAL_RTEG,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {ORIGNAL_RTEG}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


Exported 8 GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\original_rteg
Layer names: open each .gds with its matching .lyp in KLayout


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_0_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117616
1,1,shuntq3_CDNS_781040784740,rteg_1_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117634
2,2,seriesq3_CDNS_781040784741,rteg_2_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124484
3,3,seriesq3_CDNS_781040784741,rteg_3_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124484
4,4,shuntq3_CDNS_781040784742,rteg_4_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,136164
5,5,shuntq3_CDNS_781040784745,rteg_5_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,134570
6,6,seriesq3_CDNS_781040784747,rteg_6_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,122676
7,7,seriesq3_CDNS_781040784748,rteg_7_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,120844


---

## 5. Routing (MTE signal)

**~30 s read**

Build per-resonator **MTE (layer 5/0)** signal paths: collect layout roles, decide routing strategy, draw collar lip extensions, then stretch to the center pad when applicable. Exports incremental GDS after major substeps.


## 5. Routing — overview

| Step | Module | What you get |
|------|--------|--------------|
| 5.1 | `rteg_collect` | Ground plates, preserved filter metal, release holes, body MTE |
| 5.2 | `rteg_classify` | Signal vs ground nodes; `mte_route_target` |
| 5.3 | `rteg_mte_extensions` | 14 µm lip extension from collar mouth |
| 5.4 | `rteg_mte_route` | Pad stretch for `center_pad` only |
| export | `export_mte_extensions_gds` | Combined MTE (+ later MBE) GDS per index |

**Routing split:** indices **1, 3, 4, 6** → MTE to center pad · **0, 2, 5, 7** → MTE extension only (MBE signal in step 6).

**Die intercept capture (step 2.4):** mouths are read from `KB331_N_01.gds` only (filter-die coordinates). Steps 5–6 call ``DieRoutingContext.from_rteg_roles(..., die_intercepts=die_intercepts)`` to **transform** those mouths into the RTEG assembly frame (body-local offsets replayed at the RTEG metal anchor) and **snap** them onto expanded preserved collar metal. Reference RTEG GDS files are optional validation only — not production input.


### 5.1 — Collect geometry roles

**~30 s read**

**Call:** `collect_geometry_roles(assembly, res, identification, layermap)` → `all_roles[index]`

Snapshots everything routing needs in **RTEG world coordinates**: GSG pad MBE, step-4 filler plate, preserved filter interconnect (MBE/MTE), resonator body metal, release holes, inner frame boundary.

Preserved MTE includes `connectMTE` tabs; series parts may also retain the stadium-adjacent collar (e.g. index 6 → areas 911 + 5191 µm²).


In [15]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

from src.rteg_die_intercepts import DieRoutingContext

# die_routing is built after classify (step 5.0) so mouths use RTEG-frame geometry.

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

Collected geometry for 8 resonators



,index,inst_name,res_type,BAW_CAV,BAW_ReF,bottom_ground,center_node,filler_plate,frame_ring,inner_cavity,preserved_mbe,preserved_mte,top_ground,total_polygons
0,0,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,2,2,19
1,1,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,3,2,20
2,2,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,4,2,2,20
3,3,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,3,3,2,20
4,4,shuntq3_CDNS_781040784742,shunt,7,1,2,1,1,1,1,2,2,2,20
5,5,shuntq3_CDNS_781040784745,shunt,7,1,2,1,1,1,1,1,3,2,20
6,6,seriesq3_CDNS_781040784747,series,5,1,2,1,1,1,1,2,2,2,18
7,7,seriesq3_CDNS_781040784748,series,5,1,2,1,1,1,1,1,2,2,17



Polygon detail (all resonators):


,label,layer,vertices,area_um2,bbox,section,group,index,inst_name
0,pad_top[0],BAW_MBE,4,5408.0,"((120.0, 438.5), (289.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
1,pad_top[1],BAW_MBE,4,3721.0,"((59.0, 409.5), (120.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
2,pad_center[0],BAW_MBE,4,3721.0,"((59.0, 259.5), (120.0, 320.5))",ground_plates,center_node,0,shuntq3_CDNS_781040784740
3,pad_bottom[0],BAW_MBE,4,3721.0,"((59.0, 109.5), (120.0, 170.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
4,pad_bottom[1],BAW_MBE,4,5408.0,"((120.0, 109.5), (289.0, 141.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
...,...,...,...,...,...,...,...,...,...
149,BAW_CAV[3],BAW_CAV,128,132.7,"((147.0, 293.4), (160.0, 306.4))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
150,BAW_CAV[4],BAW_CAV,8,547.0,"((149.9, 274.5), (182.2, 305.3))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
151,BAW_CAV[5],BAW_CAV,125,11518.4,"((157.9, 223.8), (303.7, 322.6))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
152,inner_cavity,BAW_EDGE,4,196209.0,"((36.5, 36.5), (423.5, 543.5))",frame_boundary,inner_cavity,7,seriesq3_CDNS_781040784748


### 5.2 — Classify GSG nodes

**~30 s read**

**Calls:** `collect_orientation_inputs` → `classify_nodes` → `all_classify[index]`

Labels top/center/bottom GSG nodes as signal or ground and sets **`mte_route_target`**:

- **`center_pad`** — mouth tab is closer to the center signal pad than the body center is (route MTE to pad in 5.4).
- **`collar_extend`** — otherwise (MBE signal route in 6.1).

KB331: indices 1, 3, 4, 6 = `center_pad`; 0, 2, 5, 7 = `collar_extend`.


In [16]:
from src.rteg_classify import classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")

die_routing = DieRoutingContext.from_rteg_roles(
    all_roles,
    all_classify,
    layermap,
    res_list=res_list,
    assemblies=rteg_assemblies,
    identification=identification,
    die_intercepts=die_intercepts,
)
print(f"RTEG routing mouths ready for {len(die_routing.rteg_lips_by_index)} resonator(s)")


Classified 8 resonators



,index,inst_name,res_type,method,mte_route_target,mte_faces_center,signal_terminal,signal_drawable,collar_axis,facing_pad,top,center,bottom,note
0,0,shuntq3_CDNS_781040784740,shunt,orientation,collar_extend,False,MTE,True,east_west,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
1,1,shuntq3_CDNS_781040784740,shunt,orientation,center_pad,True,MTE,True,east_west,top,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
2,2,seriesq3_CDNS_781040784741,series,orientation,collar_extend,False,MTE,True,east_west,center,ground,ground,ground,preserved MTE not facing center — extend prese...
3,3,seriesq3_CDNS_781040784741,series,orientation,center_pad,True,MTE,True,east_west,top,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
4,4,shuntq3_CDNS_781040784742,shunt,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
5,5,shuntq3_CDNS_781040784745,shunt,orientation,collar_extend,False,MTE,True,north_south,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
6,6,seriesq3_CDNS_781040784747,series,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — route MTE to ...
7,7,seriesq3_CDNS_781040784748,series,orientation,collar_extend,False,MTE,True,east_west,top,ground,ground,ground,preserved MTE not facing center — extend prese...



Orientation classification checks passed for all 8 resonators
RTEG routing mouths ready for 8 resonator(s)


### 5.3 — MTE extensions

**~30 s read**

**Call:** `build_mte_extensions(all_roles, layermap, mte_cfg)` → one 14 µm lip extension per resonator.

Pipeline per index: pick mouth collar tab → find outward lip edge → extrude rectangle merged into collar. **`is_connected`** checks overlap + mouth coverage.

Golden baseline: **index 6** — extension on collar 911, stadium 5191. Shunt tabs: indices 0/1.

**Output:** `all_mte[index].extension` on MTE layer; tables show intercepts and connection status.


In [17]:
# DRAW MTE extensions for all resonators. 
# Disregard the orientation or position of signal in this step

from src.rteg_mte_extensions import (
    MteBuildConfig,
    build_mte_extensions,
    mte_extensions_overview_rows,
    mte_intercept_breakdown_rows,
)

# --- 5.3 tunables (edit here) ---
mte_cfg = MteBuildConfig(
    mte_layer="BAW_MTE",
    collar_extension_um=14.0,
    collar_merge_inset_um=4.0,
    collar_touch_overlap_um=0.5,
    min_collar_overlap_um2=0.01,
    stadium_collar_area_um2=2500.0,
    stadium_edge_area_ratio=0.6,
    lip_long_edge_peak_fraction=0.15,
    lip_long_edge_min_um=8.0,
    max_overlap_fraction=0.99,
    min_merge_inset_check_um=0.5,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
    feasible_merge_search_iterations=24,
)

all_mte = build_mte_extensions(all_roles, layermap, mte_cfg, die_routing=die_routing)

inst_names = {idx: roles.inst_name for idx, roles in all_roles.items()}

mte_overview_df = pd.DataFrame(
    mte_extensions_overview_rows(all_mte, inst_names=inst_names)
).sort_values("index")

display(mte_overview_df)
print(f"Drew MTE extensions for {len(all_mte)} resonators")

mte_intercept_df = pd.DataFrame(
    mte_intercept_breakdown_rows(all_mte, inst_names=inst_names)
).sort_values("index")

print("Intercept breakdown (two end-cap edges on preserved collar):")
display(mte_intercept_df)


,index,inst_name,n_preserved_mte,n_extensions,collar_overlap_um2,is_connected
0,0,shuntq3_CDNS_781040784740,2,1,11.94,False
1,1,shuntq3_CDNS_781040784740,3,1,204.78,False
2,2,seriesq3_CDNS_781040784741,2,1,151.94,True
3,3,seriesq3_CDNS_781040784741,3,1,276.54,True
4,4,shuntq3_CDNS_781040784742,2,1,376.20,True
5,5,shuntq3_CDNS_781040784745,3,1,141.49,False
6,6,seriesq3_CDNS_781040784747,2,1,703.85,False
7,7,seriesq3_CDNS_781040784748,2,1,483.98,True


Drew MTE extensions for 8 resonators
Intercept breakdown (two end-cap edges on preserved collar):


,index,inst_name,n_extensions,collar_intercept_a,collar_intercept_b,endcap_edge_a,endcap_edge_b,endcap_index_a,endcap_index_b,lip_edge_index,merge_inset_a_um,merge_inset_b_um,min_merge_um,mouth_span_um,mouth_vertices,extension_um,target_extension_um
0,0,shuntq3_CDNS_781040784740,1,"(160.46, 199.43)","(161.81, 204.42)","(160.46, 199.43) -> (161.81, 204.42)","(160.46, 199.43) -> (161.81, 204.42)",-1,-1,-1,0.68,4.00,0.68,5.17,2,14.0,14.0
1,1,shuntq3_CDNS_781040784740,1,"(181.43, 327.92)","(142.79, 341.05)","(181.43, 327.92) -> (142.79, 341.05)","(181.43, 327.92) -> (142.79, 341.05)",-1,-1,-1,0.26,0.26,0.26,40.81,2,14.0,14.0
2,2,seriesq3_CDNS_781040784741,1,"(241.94, 320.90)","(207.94, 321.30)","(241.94, 320.90) -> (207.94, 321.30)","(241.94, 320.90) -> (207.94, 321.30)",1,1,1,4.50,4.50,4.50,34.00,2,14.0,14.0
3,3,seriesq3_CDNS_781040784741,1,"(155.92, 267.31)","(156.81, 306.78)","(155.92, 267.31) -> (156.81, 306.78)","(155.92, 267.31) -> (156.81, 306.78)",-1,-1,-1,4.50,4.50,4.50,39.48,2,14.0,14.0
4,4,shuntq3_CDNS_781040784742,1,"(199.53, 253.24)","(142.56, 311.18)","(199.53, 253.24) -> (142.56, 311.18)","(199.53, 253.24) -> (142.56, 311.18)",-1,-1,-1,4.50,4.50,4.50,81.26,2,14.0,14.0
5,5,shuntq3_CDNS_781040784745,1,"(227.30, 177.01)","(224.55, 238.08)","(227.30, 177.01) -> (224.55, 238.08)","(227.30, 177.01) -> (224.55, 238.08)",-1,-1,-1,4.00,0.63,0.63,61.12,2,14.0,14.0
6,6,seriesq3_CDNS_781040784747,1,"(158.67, 248.75)","(147.97, 326.57)","(158.67, 248.75) -> (147.97, 326.57)","(158.67, 248.75) -> (147.97, 326.57)",-1,-1,-1,4.50,4.50,4.50,78.55,2,14.0,14.0
7,7,seriesq3_CDNS_781040784748,1,"(280.26, 322.59)","(174.35, 303.91)","(280.26, 322.59) -> (174.35, 303.91)","(280.26, 322.59) -> (174.35, 303.91)",36,36,36,4.50,4.50,4.50,107.55,2,14.0,14.0


### 5.3b — Die vs RTEG intercept check (pick an index)

Optional: compare step 2.4 filter-die mouth points with step 5.3/6.x collar hits after routing.

In [18]:
DEBUG_INDEX = 6  # change to inspect another resonator

from src.rteg_die_intercepts import transform_point_to_rteg

item = die_intercepts.get(DEBUG_INDEX)
lip_mte = die_routing.lip(DEBUG_INDEX, "mte")
lip_mbe = die_routing.collar_mouth(DEBUG_INDEX, "mbe")
draw = all_mte[DEBUG_INDEX].extension_draw

rows = []
for layer, li, lip in (
    ("MTE", item.mte, lip_mte),
    ("MBE", item.mbe, lip_mbe),
):
    if li is None or li.status != "ok":
        continue
    rows.append({
        "layer": layer,
        "frame": "filter_die",
        "point_a": li.intercept_a,
        "point_b": li.intercept_b,
        "mouth_span_um": li.mouth_span_um,
        "collar_area_um2": li.collar_area_um2,
    })
    if lip is not None:
        rows.append({
            "layer": layer,
            "frame": "rteg",
            "point_a": lip.point_a if hasattr(lip, "point_a") else lip[0],
            "point_b": lip.point_b if hasattr(lip, "point_b") else lip[1],
            "mouth_span_um": li.mouth_span_um,
            "collar_area_um2": li.collar_area_um2,
        })

if draw is not None:
    rows.append({
        "layer": "MTE",
        "frame": "built_5.3",
        "point_a": draw.collar_intercept_a,
        "point_b": draw.collar_intercept_b,
        "mouth_span_um": draw.mouth_span_um,
        "collar_area_um2": None,
    })

display(pd.DataFrame(rows))

,layer,frame,point_a,point_b,mouth_span_um,collar_area_um2
0,MTE,filter_die,"(225.425203693411, 244.0804384783607)","(196.6618079926439, 329.4140757812751)",90.051000,9497.3
1,MTE,rteg,"(158.66968891138706, 248.75415188478465)","(147.96880718876002, 326.5701142673805)",90.051000,9497.3
2,MBE,filter_die,"(226.49797215645197, 330.3092237173838)","(278.43100000000004, 239.1)",104.958000,9889.1
3,MBE,rteg,"(154.56478873799426, 248.72509407638404)","(139.17900000000003, 284.76700000000005)",104.958000,9889.1
4,MTE,built_5.3,"(158.66968891138706, 248.75415188478465)","(147.96880718876002, 326.5701142673805)",78.548284,NaN


### 5.3 — Export MTE extensions

**~30 s read**

**Call:** `export_mte_extensions_gds(rteg_assemblies, all_mte, MTE_OUT, layermap=...)`

Writes one GDS + `.lyp` per resonator with frame, placed geometry, and 5.3 MTE extensions (no pad stretch yet). Open in KLayout with the matching `.lyp` for layer names.

Later exports (after 5.4 / 6.x) pass `mbe_extensions` and `mbe_bodies` to add routing metal incrementally.


### 5.3 — Checkpoint

MTE extensions are built in the previous cell. GDS export is deferred to **Final export** at the end of the notebook so every step (5.4–6.3) is included in one write.


### 5.4 — Stretch MTE to center signal pad

**~30 s read**

**Call:** `build_mte_pad_routes(all_roles, all_classify, all_mte, layermap, mte_route_cfg)`

**Gate:** `mte_route_target == "center_pad"` only (KB331 indices **1, 3, 4, 6**).

Morphs the 5.3 extension outward cap to span the center signal pad height while keeping collar mouth vertices fixed. Re-export GDS in the next cell to see full MTE signal paths.

**Config:** `MteRouteConfig` — `pad_touch_overlap_um`, `min_pad_overlap_um2`.


In [19]:
from src.rteg_mte_route import (
    MteRouteConfig,
    build_mte_pad_routes,
    mte_route_overview_rows,
)

# --- 5.4 tunables (edit here) ---
mte_route_cfg = MteRouteConfig(
    mte_layer="BAW_MTE",
    pad_touch_overlap_um=0.5,
    min_pad_overlap_um2=0.01,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
)

all_mte = build_mte_pad_routes(
    all_roles, all_classify, all_mte, layermap, mte_route_cfg
)

mte_route_df = pd.DataFrame(
    mte_route_overview_rows(all_mte, all_classify, inst_names=inst_names)
).sort_values("index")

display(mte_route_df)
routed_count = int(mte_route_df["routed_to_pad"].sum())
print(f"Routed {routed_count} resonator(s) to center signal pad")


,index,inst_name,mte_route_target,mte_faces_center,routed_to_pad,pad_overlap_um2,route_width_um,n_waypoints
0,0,shuntq3_CDNS_781040784740,collar_extend,False,False,NaN,NaN,NaN
1,1,shuntq3_CDNS_781040784740,center_pad,True,True,30.3623,61.0,2.0
2,2,seriesq3_CDNS_781040784741,collar_extend,False,False,NaN,NaN,NaN
3,3,seriesq3_CDNS_781040784741,center_pad,True,True,30.4352,61.0,2.0
4,4,shuntq3_CDNS_781040784742,center_pad,True,True,30.4707,61.0,2.0
5,5,shuntq3_CDNS_781040784745,collar_extend,False,False,NaN,NaN,NaN
6,6,seriesq3_CDNS_781040784747,center_pad,True,True,30.5000,61.0,2.0
7,7,seriesq3_CDNS_781040784748,collar_extend,False,False,NaN,NaN,NaN


Routed 4 resonator(s) to center signal pad


## Step 6 — MBE routing

**~30 s read**

When MTE does **not** face the center pad (`collar_extend`), **MBE (layer 2/0)** carries signal (6.1) and ground filler (6.2). When MTE routes to the pad (`center_pad`), step **6.3** routes the step-4 ground filler around MTE keepouts instead.

| Step | Applies to | Purpose |
|------|------------|---------|
| 6.1 | 0, 2, 5, 7 | MBE curve from signal pad → preserved collar |
| 6.2 | 0, 2, 5, 7 | MBE cap + carved ground filler bridge |
| 6.3 | 1, 3, 4, 6 | Carve/route ground filler with MTE clearance |

### 6.1 — MBE signal route (pad → collar)

**Gate:** `mbe_extension_applies` → `build_mbe_extensions`

Select preserved MBE collar, ray-cast from pad corners, trace collar mouth fillet, draw `BAW_MBE` connector (`all_mbe[index].routed_net`). Re-export GDS to add MBE signal polys.


In [20]:
# MBE pad-to-collar connection for resonators where mte_faces_center = false.

from src.rteg_mbe_extensions import (
    MbeConnectionConfig,
    build_mbe_extensions,
    mbe_extensions_overview_rows,
)

mbe_cfg = MbeConnectionConfig()

all_mbe = build_mbe_extensions(all_roles, all_classify, layermap, mbe_cfg, die_routing=die_routing)

mbe_overview_df = pd.DataFrame(
    mbe_extensions_overview_rows(all_mbe, inst_names=inst_names)
).sort_values("index")

display(mbe_overview_df)
mbe_drawn = int(mbe_overview_df["n_extensions"].sum())
print(f"Drew MBE connections for {mbe_drawn} resonators (mte_faces_center = false)")


,index,inst_name,n_preserved_mbe,n_extensions,point_a,point_b,hit_a,hit_b,area_um2,drc_violations
0,0,shuntq3_CDNS_781040784740,3,1,"(119.98, 320.50)","(119.98, 259.50)","(113.97, 273.19)","(166.46, 211.98)",600.01,None
1,1,shuntq3_CDNS_781040784740,3,0,NaN,NaN,NaN,NaN,0.00,None
2,2,seriesq3_CDNS_781040784741,4,1,"(119.98, 320.50)","(119.98, 259.50)","(148.84, 304.22)","(165.17, 261.95)",1594.04,None
3,3,seriesq3_CDNS_781040784741,3,0,NaN,NaN,NaN,NaN,0.00,None
4,4,shuntq3_CDNS_781040784742,2,0,NaN,NaN,NaN,NaN,0.00,None
5,5,shuntq3_CDNS_781040784745,1,1,"(119.98, 320.50)","(119.98, 259.50)","(155.96, 316.73)","(159.80, 293.30)",1628.59,None
6,6,seriesq3_CDNS_781040784747,2,0,NaN,NaN,NaN,NaN,0.00,None
7,7,seriesq3_CDNS_781040784748,1,1,"(119.98, 320.50)","(119.98, 259.50)","(154.20, 263.13)","(183.27, 221.25)",1087.49,None


Drew MBE connections for 4 resonators (mte_faces_center = false)


## Step 6.2 — MBE ground body (`collar_extend` only)

**~30 s read**

**Gate:** `mbe_body_collar_extend_applies` → `build_mbe_body_collar_extends` — KB331 indices **0, 2, 5, 7**.

1. **MBE cap** on outer half of 5.3 MTE extension (shifted 3.5 µm outward onto filler).
2. **Keepouts** — body MTE stadium, release holes, 5.3 MTE extension, 6.1 MBE signal; boolean-carve step-4 filler.
3. **Bridge** — reconnect carved filler to cap across stadium gap.
4. **Export** — cap + filler pieces as separate `BAW_MBE` polygons; raw step-4 rectangle stripped.

Run **Final export** (last notebook cell) after step 6.3 to write complete GDS files.


In [21]:
# MBE ground body for resonators where mte_route_target = collar_extend.

from src.rteg_mbe_body import (
    MbeBodyConfig,
    build_mbe_body_collar_extends,
    mbe_body_overview_rows,
)

COLLAR_EXTEND_INDICES = (0, 2, 5, 7)

mbe_body_cfg = MbeBodyConfig()

all_mbe_body = build_mbe_body_collar_extends(
    all_roles,
    all_classify,
    all_mte,
    all_mbe,
    layermap,
    mbe_body_cfg,
)

mbe_body_overview_df = pd.DataFrame(
    mbe_body_overview_rows(all_mbe_body, inst_names=inst_names)
).sort_values("index")

collar_extend_df = mbe_body_overview_df[
    mbe_body_overview_df["index"].isin(COLLAR_EXTEND_INDICES)
]
display(collar_extend_df)
body_drawn = int(collar_extend_df["n_pieces"].sum())
print(f"Drew MBE ground body for {body_drawn} polygon piece(s) (collar_extend)")


,index,inst_name,n_pieces,cap_area_um2,filler_area_um2,drc_violations
0,0,shuntq3_CDNS_781040784740,3,46.54,46168.98,could not build filler bridge from MBE cap to ...
1,2,seriesq3_CDNS_781040784741,2,306.02,62407.76,NaN
2,5,shuntq3_CDNS_781040784745,2,447.25,41885.04,could not build filler bridge from MBE cap to ...
3,7,seriesq3_CDNS_781040784748,3,967.91,53195.36,NaN


Drew MBE ground body for 10 polygon piece(s) (collar_extend)


## Step 6.3 — MBE ground filler (`center_pad` only)

**~30 s read**

**Gate:** `mbe_body_center_pad_applies` → `build_mbe_body_center_pads` — KB331 indices **1, 3, 4, 6**.

Carves the step-4 **MBE width filler** with clearance to MTE body, 5.3 extension, and pad route (layer 5/0). Left edge follows the preserved MBE collar; top/right/bottom are trimmed back from MTE routes (~2 µm + 1 µm trim margin) so no pinch corners sit on the extension-to-pad junction.

**Output:** one connected `BAW_MBE` filler polygon per index (collar metal absorbed into filler export).

Results merge into `all_mbe_body` via `merge_mbe_bodies` (6.2 and 6.3 cover disjoint index sets).


In [22]:
# Step 6.3 — center_pad resonators: route the step-4 MBE filler rectangle
# around the stadium by hugging the outer edge of the preserved MBE collar.

from src.rteg_mbe_body_center_pad import (
    MbeBodyCenterPadConfig,
    build_mbe_body_center_pads,
    mbe_body_center_pad_overview_rows,
)

center_pad_cfg = MbeBodyCenterPadConfig()

center_pad_body = build_mbe_body_center_pads(
    all_roles,
    all_classify,
    all_mte,
    layermap,
    center_pad_cfg,
    die_routing=die_routing,
)

from src.rteg_mbe_body import merge_mbe_bodies

all_mbe_body = merge_mbe_bodies(all_mbe_body, center_pad_body)

center_pad_df = pd.DataFrame(
    mbe_body_center_pad_overview_rows(center_pad_body, inst_names=inst_names)
).sort_values("index")
display(center_pad_df)
body_drawn = int(center_pad_df["n_pieces"].sum())
print(f"Drew MBE ground filler for {body_drawn} polygon piece(s) (center_pad)")


,index,inst_name,n_pieces,filler_area_um2,drc_violations
0,1,shuntq3_CDNS_781040784740,1,50972.87,None
1,3,seriesq3_CDNS_781040784741,1,65958.66,None
2,4,shuntq3_CDNS_781040784742,1,52337.64,None
3,6,seriesq3_CDNS_781040784747,1,64631.25,None


Drew MBE ground filler for 4 polygon piece(s) (center_pad)


## Final export — complete routed RTEG (steps 4–6.3)

**~30 s read**

**Call:** `export_full_rteg_gds(rteg_assemblies, all_mte, FINAL_OUT, layermap=..., parent=..., mbe_extensions=all_mbe, mbe_bodies=all_mbe_body)`

Run this cell **after** steps 6.2 and 6.3 so `all_mbe_body` includes both routing styles (collar_extend + center_pad).

**Output:** `draft_output/final_rteg/{parent}_RTEG1_{index:02d}_{inst_name}_routed.gds` — one file per resonator with die frame, PPD, resonator, MTE routes, and MBE signal/ground from every pipeline step. Open in KLayout with the sidecar `.lyp`.


In [23]:
# Final export — all pipeline geometry in one GDS per resonator.

from src.export_gds import export_summary_df
from src.rteg_mte_extensions import export_full_rteg_gds

FINAL_OUT = DRAFT / "final_rteg"
final_export_df = export_summary_df(
    export_full_rteg_gds(
        rteg_assemblies,
        all_mte,
        FINAL_OUT,
        layermap=layermap,
        parent=parent,
        mbe_extensions=all_mbe,
        mbe_bodies=all_mbe_body,
    )
)

print(f"Exported {len(final_export_df)} complete RTEG GDS files to {FINAL_OUT}")
display(final_export_df)


Exported 8 complete RTEG GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\final_rteg


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_0_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117894
1,1,shuntq3_CDNS_781040784740,rteg_1_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,118448
2,2,seriesq3_CDNS_781040784741,rteg_2_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,125088
3,3,seriesq3_CDNS_781040784741,rteg_3_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,123640
4,4,shuntq3_CDNS_781040784742,rteg_4_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,136018
5,5,shuntq3_CDNS_781040784745,rteg_5_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,135000
6,6,seriesq3_CDNS_781040784747,rteg_6_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,123136
7,7,seriesq3_CDNS_781040784748,rteg_7_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,121536
